# Playbook: Deep Research (SGR Agent Core)

## 1. Клонирование и переход в репозиторий

In [1]:
!test -d sgr-agent-core || git clone https://github.com/vamplabai/sgr-agent-core.git
%cd sgr-agent-core

/storage/Repository/sgr/aiconf/sgr-agent-core


## 2. Виртуальное окружение и установка пакета

In [2]:
!test -d .venv || python3 -m venv .venv
!source .venv/bin/activate
!pip install -q -U pip
!pip install -q -e .

## 3. Каталоги `logs/` и `reports/`

In [3]:
!mkdir -p logs reports
!ls -la logs reports

logs:
итого 8
drwxrwxr-x  2 pasha pasha 4096 апр 12 23:59 .
drwxrwxr-x 15 pasha pasha 4096 апр 12 23:59 ..

reports:
итого 8
drwxrwxr-x  2 pasha pasha 4096 апр 12 23:59 .
drwxrwxr-x 15 pasha pasha 4096 апр 12 23:59 ..


## 4. Конфиг из примера

In [4]:
# Копируем конфиг
!cp examples/sgr_deep_research/config.yaml.example examples/sgr_deep_research/config.yaml

# Правил настройки
!sed -i \
  -e 's|your-openai-api-key-here|https://t.me/evilfreelancer|g' \
  -e 's|https://api.openai.com/v1|https://api.rpa.icu/v1|g' \
  -e 's|gpt-4.1-mini|gpt-oss:120b|g' \
  -e 's|your-tavily-api-key-here|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|g' \
  examples/sgr_deep_research/config.yaml

# Читаем что получилось
!head -45 examples/sgr_deep_research/config.yaml

# SGR Deep Research - Configuration for research agents
# Copy this file and fill in your API keys

# LLM Configuration
llm:
  api_key: "https://t.me/evilfreelancer"  # Your OpenAI API key
  base_url: "https://api.rpa.icu/v1"  # API base URL
  model: "gpt-oss:120b"  # Model name
  max_tokens: 8000  # Max output tokens
  temperature: 0.4  # Temperature (0.0-1.0)
  # proxy: "socks5://127.0.0.1:1081"  # Optional proxy (socks5:// or http://)

# Execution Settings
execution:
  max_clarifications: 3  # Max clarification requests
  max_iterations: 10  # Max agent iterations
  mcp_context_limit: 15000  # Max context length from MCP server response
  streaming_generator: "openai"  # OpenAI SSE format; use "open_webui" for Open WebUI <details> format
  logs_dir: "logs"  # Directory for saving agent execution logs
  reports_dir: "reports"  # Directory for saving research reports

# MCP (Model Context Protocol) Configuration
mcp:
  mcpServers:
    deepwiki:
      url: "https://mcp.deepwiki.com/mcp

## 5. Запуск API (порт 8012)

In [5]:
import os
get_ipython().system = os.system
!nohup sgr -c examples/sgr_deep_research/config.yaml -p 8012 >> logs/sgr_agent_core.log &
!sleep 15
!curl -s -S http://localhost:8012/v1/models | python3 -m json.tool

INFO:     Started server process [1135302]
INFO:     Waiting for application startup.
2026-04-12 23:59:28,397 - sgr_agent_core.services.overlayfs_manager - 237 - ERROR - Failed to initialize OverlayFS for RunCommandTool
Traceback (most recent call last):
  File "/storage/Repository/sgr/aiconf/sgr-agent-core/sgr_agent_core/services/overlayfs_manager.py", line 182, in initialize_from_config
    candidates = _get_run_command_config_candidates(config)
  File "/storage/Repository/sgr/aiconf/sgr-agent-core/sgr_agent_core/services/overlayfs_manager.py", line 39, in _get_run_command_config_candidates
    _, tool_configs = AgentFactory._resolve_tools_with_configs(agent_def.tools, config)
                      ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: AgentFactory._resolve_tools_with_configs() takes 2 positional arguments but 3 were given
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8012 (Press CTRL+C to quit)


{
    "data": [
        {
            "id": "sgr_agent",
            "object": "model",
            "created": 1234567890,
            "owned_by": "sgr-agent-core"
        },
        {
            "id": "tool_calling_agent",
            "object": "model",
            "created": 1234567890,
            "owned_by": "sgr-agent-core"
        },
        {
            "id": "sgr_tool_calling_agent",
            "object": "model",
            "created": 1234567890,
            "owned_by": "sgr-agent-core"
        },
        {
            "id": "iron_agent",
            "object": "model",
            "created": 1234567890,
            "owned_by": "sgr-agent-core"
        },
        {
            "id": "dialog_agent",
            "object": "model",
            "created": 1234567890,
            "owned_by": "sgr-agent-core"
        }
    ],
    "object": "list"
}


0

## 6. Запрос типа Deep Research

Просим агента найти и расписат файты про питоновский asyncio.

In [6]:
!curl -N -m 180 -sS -X POST http://localhost:8012/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"tool_calling_agent","stream":true,"messages":[{"role":"user","content":"Do a very short research summary (3-5 bullet facts) about the Python asyncio event loop. You must use the create report tool and write a markdown file under the reports/ directory."}]}'

data: {"id": "0-start", "object": "chat.completion.chunk", "created": 1776027581, "model": "tool_calling_agent_0f6a4abf-6052-4cd1-9eed-82d21d239f78", "system_fingerprint": null, "choices": [{"delta": {"content": "Agent tool_calling_agent_0f6a4abf-6052-4cd1-9eed-82d21d239f78 started\n", "role": "assistant", "tool_calls": null}, "index": 0, "finish_reason": null, "logprobs": null}], "usage": null}

data: {"id":"1-action","choices":[{"delta":{"content":null,"function_call":null,"refusal":null,"role":"assistant","tool_calls":null,"reasoning_content":"","reasoning":""},"finish_reason":null,"index":0,"logprobs":null}],"created":1776027582,"model":"tool_calling_agent_0f6a4abf-6052-4cd1-9eed-82d21d239f78","object":"chat.completion.chunk","service_tier":null,"system_fingerprint":null,"usage":null}

data: {"id":"1-action","choices":[{"delta":{"content":null,"function_call":null,"refusal":null,"role":null,"tool_calls":null,"reasoning_content":"The","reasoning":"The"},"finish_reason":null,"index":

2026-04-13 00:00:13,338 - sgr_agent_core.agents.tool_calling_agent_0f6a4abf-6052-4cd1-9eed-82d21d239f78 - 290 - ERROR - ❌ Agent execution error: litellm.ServiceUnavailableError: litellm.MidStreamFallbackError: litellm.APIConnectionError: APIConnectionError: OpenAIException - Unknown role: assistant<|channel|>commentary. Received Model Group=gpt-oss:120b
Available Model Group Fallbacks=None
Traceback (most recent call last):
  File "/storage/Repository/sgr/aiconf/sgr-agent-core/sgr_agent_core/base_agent.py", line 281, in _execute
    await self._execution_step()
  File "/storage/Repository/sgr/aiconf/sgr-agent-core/sgr_agent_core/base_agent.py", line 231, in _execution_step
    action_tool = await self._select_action_phase(reasoning)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/storage/Repository/sgr/aiconf/sgr-agent-core/sgr_agent_core/agents/tool_calling_agent.py", line 49, in _select_action_phase
    async for event in stream:
        if event.type == "chunk"

0

## 7. Проверка отчёта

In [7]:
!ls -la reports/
!echo "--- latest report (first 80 lines) ---"
!test -n "$(ls reports/*.md 2>/dev/null)" && head -80 "$(ls -t reports/*.md | head -1)" || echo "No .md files in reports/"

итого 12
drwxrwxr-x  2 pasha pasha 4096 апр 13 00:00 .
drwxrwxr-x 15 pasha pasha 4096 апр 12 23:59 ..
-rw-rw-r--  1 pasha pasha 1549 апр 13 00:00 20260413_000007_Python asyncio event loop  short research summary.md
--- latest report (first 80 lines) ---
# Python asyncio event loop – short research summary

*Created: 2026-04-13 00:00:07*

# Python asyncio event loop – short research summary

- The event loop is the core of every asyncio application, executing asynchronous tasks, callbacks, network I/O operations, and subprocesses. [1]
- Developers are encouraged to use high‑level helpers such as `asyncio.run()` and usually do not need to manipulate the loop object directly. [1]
- An event loop runs in a single thread (typically the main thread) and schedules callbacks and `Task` objects from its internal queue. [1]
- The asyncio module provides low‑level functions to obtain, set, or create an event loop, e.g., `asyncio.get_running_loop()` and `asyncio.new_event_loop()`. [1]
- The loop d

0

## 8. Остановка сервера

In [8]:
!pkill -f "sgr.*8012" 2>/dev/null || true
!echo done

done


0